In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("GEMINI_API_KEY") is not None)

True


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [5]:
rag implementation


SyntaxError: invalid syntax (4146192373.py, line 1)

In [6]:
from langchain_core.documents import Document

my_text = """The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation."""

docs = [Document(page_content=my_text,metadata ={"source":"ABC","id":"1"} )]

docs

[Document(metadata={'source': 'ABC', 'id': '1'}, page_content="The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]\n\nArtificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Fun

STEP 2: CHUNKING

TO PREPARE THE CHUNKS we can use langchain - recursive character text splitter

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(

chunk_size = 500,
chunk_overlap = 50

)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'id': '1'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2]'),
 Document(metadata={'source': 'ABC', 'id': '1'}, page_content='philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]'),
 Document(metadata={'source': 'ABC', 'id': '1'}, page_content='Artificial intelligence was founded as an academic discipline in 195

In [9]:
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Number of chunks: 4

--- Chunk 1 ---
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2]

--- Chunk 2 ---
philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

--- Chunk 3 ---
Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment

STEP3: CREATING EMBEDDING FOR THE CHUNKS

OPENAI EMBEDDINGS MODEL

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [ ]:
embeddings.embed_query("What is AI")
#simple embedding

[-0.016259853,
 -0.0043472173,
 0.003663936,
 -0.05135757,
 -0.01767093,
 -0.0037210095,
 0.018771552,
 -0.0066824225,
 0.024736216,
 -0.013018557,
 -0.007996252,
 -0.008031463,
 0.0011244181,
 0.032236103,
 0.10273968,
 0.005347275,
 0.024449237,
 -0.0014507688,
 0.00353001,
 -0.018742196,
 0.010785148,
 -0.008917651,
 -0.006632313,
 0.0028245032,
 -0.018053917,
 -0.027268376,
 -0.003540356,
 0.002934898,
 0.019941628,
 0.0006659874,
 -0.0012688489,
 -0.028184228,
 0.017142696,
 0.024109595,
 0.0067620277,
 0.0043110834,
 0.012547365,
 -0.017924715,
 0.000904636,
 0.044851895,
 -0.020457795,
 -0.017764004,
 -0.019728074,
 -0.015671277,
 -0.0049128807,
 -0.0015374831,
 0.01770253,
 -0.035637174,
 -0.032405432,
 0.017852383,
 -0.0038905598,
 -0.006584692,
 -0.0016654152,
 -0.15918015,
 0.0008999293,
 0.0063361074,
 -0.003082571,
 0.002483074,
 -0.018848224,
 -0.028859193,
 -0.0302302,
 -0.014695832,
 -0.00043466766,
 -0.020286696,
 0.009092669,
 0.019210173,
 0.017175827,
 0.028216315,


STEP4: CREATE AND STORE EMBEDDING INTO VECTOR STORE

chroma, faiss,pinecone,qdrant is a vector database 

We give LangChain the chunks and embedding model → LangChain converts the chunks into vectors → Chroma stores the vectors along with the corresponding documents.

In [14]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
#MANUAL WAY OF CONVERTING CHUNKS INTO EMBEDDINGS
vectors = []

#Loops through every chunk one by one.
for doc in chunks:
    vector = embeddings.embed_documents([doc.page_content])
    vectors.append(vector)

In [ ]:
vectorstore.similarity_search("What is AI",k=3)

# first what is ai is converted into vector using model
#this vector willl go to vector store and similarity search will be done
#and it will return top 3 vectors with mapped chunks
#this will be come to user

[Document(id='6ad3a1a8-a56b-4f73-8192-d11c5cf2c43d', metadata={'id': '1', 'source': 'ABC'}, page_content='philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]'),
 Document(id='2d09bd42-88e3-431a-8536-1cee7a77251b', metadata={'id': '1', 'source': 'ABC'}, page_content='Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the'),
 Document(id='d86eb3de-a3b2-46ec-ab1d-dd0d987877e

talk to LLM


In [ ]:
context = vectorstore.similarity_search("What is AI",k=3)

#generates top 3 relevant chunks


In [ ]:
res = llm.invoke(f"what is ai:{context}")
print(res.content)

#model generates answer

d:\rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Based on the provided documents, **Artificial Intelligence (AI)** is an interdisciplinary field founded as an academic discipline in 1956 that focuses on creating systems capable of performing tasks that typically require human-like cognitive abilities.\n\nHere are the key aspects of AI outlined in the text:\n\n* **Core Goals:** Research in AI focuses on enabling machines to perform learning, reasoning, knowledge representation, planning, natural language processing (NLP), perception, and robotics.\n* **Key Techniques:** Researchers achieve these goals using techniques such as artificial neural networks, deep learning, statistical methods, formal logic, state space search, and mathematical optimization.\n* **Interdisciplinary Nature:** AI draws heavily upon various fields, including computer science, neuroscience, psychology, linguistics, philosophy, economics, and operations research.\n* **Evolution & Recent Progress:** After cycles of optimism and periods o